# Spectrogram-Based Vishing Detector

## Imports and Configurations

In [ ]:
from pathlib import Path
import sys

import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

import math

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from modules.dataset_processing import (
    WaveformDataset,
    extract_archives,
    prepare_all_split_dataframes,
    subsample_by_class,
 )
from modules.attacks import FGSMAttack, FGSMAttackConfig
from modules.audio_processing import LogMelSpectrogramConfig
from modules.models import SpectrogramCNNConfig, SpectrogramClassifier

In [ ]:
DATA_ROOT = PROJECT_ROOT / 'ASVspoof5'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = MODEL_DIR / 'spectrogram'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TEST_MODE = False
BONAFIDE_SUBSAMPLE = 1200
SPOOF_SUBSAMPLE = 1800
TARGET_NUM_SAMPLES = 64000
BATCH_SIZE = 32
NUM_EPOCHS = 70
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
RANDOM_SEED = 67
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EARLY_STOPPING_PATIENCE = 4

torch.manual_seed(RANDOM_SEED)
print({'device': str(DEVICE), 'data_root': str(DATA_ROOT)})

## Data Preprocessing

In [ ]:
extract_archives(DATA_ROOT)
split_frames = prepare_all_split_dataframes(DATA_ROOT, require_existing_files=True)

if TEST_MODE:
    split_frames['train'] = subsample_by_class(split_frames['train'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)
    split_frames['dev'] = subsample_by_class(split_frames['dev'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)
    split_frames['eval'] = subsample_by_class(split_frames['eval'], BONAFIDE_SUBSAMPLE, SPOOF_SUBSAMPLE, random_seed=RANDOM_SEED)

for split_name, frame in split_frames.items():
    print(split_name, frame.shape, frame['LABEL'].value_counts().to_dict())

In [ ]:
for split_name, frame in split_frames.items():
    counts = frame['LABEL'].value_counts().to_dict()
    total = sum(counts.values())
    print(f"{split_name}: {counts} | bonafide ratio: {counts.get(0,0)/total:.3f}")
print(split_frames['train']['ATTACK_LABEL'].value_counts())

In [ ]:
train_dataset = WaveformDataset(
    split_frames['train'],
    sample_rate=16000,
    target_num_samples=TARGET_NUM_SAMPLES,
    random_crop=True,
)
dev_dataset = WaveformDataset(
    split_frames['dev'],
    sample_rate=16000,
    target_num_samples=TARGET_NUM_SAMPLES,
    random_crop=False,
)
eval_dataset = WaveformDataset(
    split_frames['eval'],
    sample_rate=16000,
    target_num_samples=TARGET_NUM_SAMPLES,
    random_crop=False,
)

# # Change all DataLoaders from num_workers=0
# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
# dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False, num_workers=1, pin_memory=False)

## No Training

In [ ]:
frontend_config = LogMelSpectrogramConfig(sample_rate=16000, n_fft=1024, hop_length=256, n_mels=80, f_min=20.0, f_max=7600.0)
model_config = SpectrogramCNNConfig(
    num_classes=9,
    base_channels=32,   # was 128 
    dropout=0.3,        # was 0.4
    embedding_dim=256   # was 512
)
model = SpectrogramClassifier(
    transformer_config=frontend_config,
    model_config=model_config,
    use_spec_augment=False,
).to(DEVICE)

In [ ]:
# Load best loss checkpoint
best_checkpoint = torch.load(CHECKPOINT_DIR / 'best_loss.pt', weights_only=False, map_location=DEVICE)
model.load_state_dict(best_checkpoint['model_state_dict'])
print(f"Loaded best loss model from epoch {best_checkpoint['epoch']}")

## Evaluation

In [ ]:
fgsm_attack = FGSMAttack(
    config=FGSMAttackConfig(
        epsilon=0.002,
        clamp_min=-1.0,
        clamp_max=1.0,
        targeted=False,
    )
)
all_labels = []
all_clean_predictions = []
all_adversarial_predictions = []
all_clean_true_probabilities = []
all_adversarial_true_probabilities = []

model.eval()
for batch in eval_loader:
    waveforms = batch['waveform'].to(DEVICE)
    labels = batch['label'].to(DEVICE)

    with torch.no_grad():
        clean_logits = model(waveforms)
        clean_probabilities = torch.softmax(clean_logits, dim=1)
        clean_predictions = clean_probabilities.argmax(dim=1)
        clean_bonafide_probabilities = clean_probabilities[:, 0]

    attack_result = fgsm_attack.generate(model=model, inputs=waveforms, labels=labels)
    adversarial_waveforms = attack_result.adversarial_inputs

    with torch.no_grad():
        adversarial_logits = model(adversarial_waveforms)
        adversarial_probabilities = torch.softmax(adversarial_logits, dim=1)
        adversarial_predictions = adversarial_probabilities.argmax(dim=1)
        adversarial_bonafide_probabilities = adversarial_probabilities[:, 0]

    all_labels.append(labels.detach().cpu())
    all_clean_predictions.append(clean_predictions.detach().cpu())
    all_adversarial_predictions.append(adversarial_predictions.detach().cpu())
    all_clean_true_probabilities.append(clean_bonafide_probabilities.detach().cpu())
    all_adversarial_true_probabilities.append(adversarial_bonafide_probabilities.detach().cpu())

all_labels = torch.cat(all_labels)
all_clean_predictions = torch.cat(all_clean_predictions)
all_adversarial_predictions = torch.cat(all_adversarial_predictions)
all_clean_true_probabilities = torch.cat(all_clean_true_probabilities)
all_adversarial_true_probabilities = torch.cat(all_adversarial_true_probabilities)

print({'num_samples': int(all_labels.shape[0])})

### Accuracy

### Mean Squared Confidence Degradation (MSCD)

### Attack Success Rate (ASR)

In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

def compute_eer(labels, scores):
    """
    labels: 0 = bonafide, 1 = spoof
    scores: model's probability of being BONAFIDE (class 0)
    EER is where FAR == FRR
    """
    # sklearn's roc_curve treats 1 as positive
    # We want bonafide (0) as the "positive" class for ASV context
    # So flip: use probability of bonafide, and flip labels
    bonafide_scores = scores  # probability assigned to bonafide class
    bonafide_labels = (labels == 0).astype(int)

    fpr, tpr, thresholds = roc_curve(bonafide_labels, bonafide_scores)
    
    fnr = 1 - tpr  # False Negative Rate = FRR
    # EER is where FPR (FAR) == FNR (FRR)
    eer_index = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[eer_index] + fnr[eer_index]) / 2
    eer_threshold = thresholds[eer_index]
    
    return eer, eer_threshold

In [ ]:
# clean_accuracy = (all_clean_predictions == all_labels).float().mean().item()
# adversarial_accuracy = (all_adversarial_predictions == all_labels).float().mean().item()

# print(f"Clean Accuracy: {clean_accuracy:.4f}")
# print(f"Adversarial Accuracy: {adversarial_accuracy:.4f}")

# delta_probabilities = all_clean_true_probabilities - all_adversarial_true_probabilities
# mscd = (delta_probabilities ** 2).mean().item()

# print(f"MSCD: {mscd:.6f}")

# asr = (all_clean_predictions != all_adversarial_predictions).float().mean().item()

# print(f"ASR: {asr:.4f}")

# # Convert to numpy for sklearn
# labels_np = all_labels.numpy()
# clean_probs_np = all_clean_true_probabilities.numpy()
# adversarial_probs_np = all_adversarial_true_probabilities.numpy()

# # Clean EER
# clean_eer, clean_threshold = compute_eer(labels_np, clean_probs_np)
# print(f"Clean EER:       {clean_eer * 100:.2f}%  (threshold: {clean_threshold:.4f})")

# # Adversarial EER
# adv_eer, adv_threshold = compute_eer(labels_np, adversarial_probs_np)
# print(f"Adversarial EER: {adv_eer * 100:.2f}%  (threshold: {adv_threshold:.4f})")

In [ ]:
binary_labels = (all_labels != 0).long()
binary_clean_preds = (all_clean_predictions != 0).long()
binary_adv_preds = (all_adversarial_predictions != 0).long()

clean_accuracy = (binary_clean_preds == binary_labels).float().mean().item()
adversarial_accuracy = (binary_adv_preds == binary_labels).float().mean().item()

print(f"Clean Accuracy: {clean_accuracy:.4f}")
print(f"Adversarial Accuracy: {adversarial_accuracy:.4f}")

delta_probabilities = all_clean_true_probabilities - all_adversarial_true_probabilities
mscd = (delta_probabilities ** 2).mean().item()

print(f"MSCD: {mscd:.6f}")

asr = (binary_clean_preds != binary_adv_preds).float().mean().item()

print(f"ASR: {asr:.4f}")

# Convert to numpy for sklearn
labels_np = binary_labels.numpy()
clean_probs_np = all_clean_true_probabilities.numpy()
adversarial_probs_np = all_adversarial_true_probabilities.numpy()

# Clean EER
clean_eer, clean_threshold = compute_eer(labels_np, clean_probs_np)
print(f"Clean EER:       {clean_eer * 100:.2f}%  (threshold: {clean_threshold:.4f})")

# Adversarial EER
adv_eer, adv_threshold = compute_eer(labels_np, adversarial_probs_np)
print(f"Adversarial EER: {adv_eer * 100:.2f}%  (threshold: {adv_threshold:.4f})")

In [ ]:
del all_labels, all_clean_predictions, all_adversarial_predictions
del all_clean_true_probabilities, all_adversarial_true_probabilities
torch.cuda.empty_cache() if torch.cuda.is_available() else None
import gc; gc.collect()

In [ ]:
# Load best EER checkpoint
best_eer_checkpoint = torch.load(CHECKPOINT_DIR / 'best_eer.pt', weights_only=False, map_location=DEVICE)
model.load_state_dict(best_eer_checkpoint['model_state_dict'])
print(f"Loaded best EER model from epoch {best_eer_checkpoint['epoch']}")

In [ ]:
fgsm_attack = FGSMAttack(
    config=FGSMAttackConfig(
        epsilon=0.002,
        clamp_min=-1.0,
        clamp_max=1.0,
        targeted=False,
    )
)
all_labels = []
all_clean_predictions = []
all_adversarial_predictions = []
all_clean_true_probabilities = []
all_adversarial_true_probabilities = []

model.eval()
for batch in eval_loader:
    waveforms = batch['waveform'].to(DEVICE)
    labels = batch['label'].to(DEVICE)

    with torch.no_grad():
        clean_logits = model(waveforms)
        clean_probabilities = torch.softmax(clean_logits, dim=1)
        clean_predictions = clean_probabilities.argmax(dim=1)
        clean_bonafide_probabilities = clean_probabilities[:, 0]

    attack_result = fgsm_attack.generate(model=model, inputs=waveforms, labels=labels)
    adversarial_waveforms = attack_result.adversarial_inputs

    with torch.no_grad():
        adversarial_logits = model(adversarial_waveforms)
        adversarial_probabilities = torch.softmax(adversarial_logits, dim=1)
        adversarial_predictions = adversarial_probabilities.argmax(dim=1)
        adversarial_bonafide_probabilities = adversarial_probabilities[:, 0]

    all_labels.append(labels.detach().cpu())
    all_clean_predictions.append(clean_predictions.detach().cpu())
    all_adversarial_predictions.append(adversarial_predictions.detach().cpu())
    all_clean_true_probabilities.append(clean_bonafide_probabilities.detach().cpu())
    all_adversarial_true_probabilities.append(adversarial_bonafide_probabilities.detach().cpu())

all_labels = torch.cat(all_labels)
all_clean_predictions = torch.cat(all_clean_predictions)
all_adversarial_predictions = torch.cat(all_adversarial_predictions)
all_clean_true_probabilities = torch.cat(all_clean_true_probabilities)
all_adversarial_true_probabilities = torch.cat(all_adversarial_true_probabilities)

binary_labels = (all_labels != 0).long()                          # 0=bonafide, 1=any attack
binary_clean_preds = (all_clean_predictions != 0).long()          # collapse A01-A08  spoof
binary_adv_preds = (all_adversarial_predictions != 0).long()


print({'num_samples': int(all_labels.shape[0])})

In [ ]:

clean_accuracy = (binary_clean_preds == binary_labels).float().mean().item()
adversarial_accuracy = (binary_adv_preds == binary_labels).float().mean().item()

print(f"Clean Accuracy: {clean_accuracy:.4f}")
print(f"Adversarial Accuracy: {adversarial_accuracy:.4f}")

delta_probabilities = all_clean_true_probabilities - all_adversarial_true_probabilities
mscd = (delta_probabilities ** 2).mean().item()

print(f"MSCD: {mscd:.6f}")

asr = (binary_clean_preds != binary_adv_preds).float().mean().item()

print(f"ASR: {asr:.4f}")

# Convert to numpy for sklearn
labels_np = binary_labels.numpy()
clean_probs_np = all_clean_true_probabilities.numpy()
adversarial_probs_np = all_adversarial_true_probabilities.numpy()

# Clean EER
clean_eer, clean_threshold = compute_eer(labels_np, clean_probs_np)
print(f"Clean EER:       {clean_eer * 100:.2f}%  (threshold: {clean_threshold:.4f})")

# Adversarial EER
adv_eer, adv_threshold = compute_eer(labels_np, adversarial_probs_np)
print(f"Adversarial EER: {adv_eer * 100:.2f}%  (threshold: {adv_threshold:.4f})")